<a href="https://colab.research.google.com/github/Ruba-Alharbi/MentalQA_NER-Reproduction/blob/main/RP1_(F)_LLM_as_a_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data merging and cleaning

### Cleaning

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
# Build the full few-shot input file for LLM-as-a-Judge.

import pandas as pd

gpt_llama = pd.read_excel("all_models_fewshot_llama_retried.xlsx")
allam = pd.read_excel("few-allam_df.xlsx")

gpt_llama.columns = gpt_llama.columns.str.strip()
allam.columns = allam.columns.str.strip()

gpt_llama = gpt_llama[
    ["question", "GPT-4o Output", "LLaMA Output"]
].copy()

allam = allam[
    ["question", "Allam Output"]
].copy()

all_models_fewshot = gpt_llama.merge(
    allam,
    on="question",
    how="left"
)

print(f"Rows: {len(all_models_fewshot)}")
print(
    all_models_fewshot[
        ["GPT-4o Output", "LLaMA Output", "Allam Output"]
    ].isna().sum()
)

all_models_fewshot.to_excel(
    "all_models_fewshot.xlsx",
    index=False
)

print("Saved all_models_fewshot.xlsx")

Rows: 499
GPT-4o Output    0
LLaMA Output     0
Allam Output     0
dtype: int64
Saved all_models_fewshot.xlsx


In [ ]:
import pandas as pd
all_models = pd.read_excel('all_models_fewshot.xlsx')

In [ ]:
all_models.columns

Index(['question', 'GPT-4o Output', 'LLaMA Output', 'Allam Output'], dtype='object')

In [ ]:
# Identify LLaMA API failures that were incorrectly saved as model outputs.

import pandas as pd

judge_input = pd.read_excel("all_models_fewshot.xlsx")

llama_failure_mask = (
    judge_input["LLaMA Output"]
    .fillna("")
    .astype(str)
    .str.startswith("LLaMA Error:")
)

failed_llama_questions = judge_input.loc[
    llama_failure_mask,
    ["question", "LLaMA Output"]
].copy()

print(f"Failed LLaMA calls: {len(failed_llama_questions)} / {len(judge_input)}")

display(failed_llama_questions)

Failed LLaMA calls: 0 / 499


,question,LLaMA Output


In [ ]:
# Retry only LLaMA outputs that failed because of temporary API rate limits.
# Successful model outputs are preserved without modification.

import pandas as pd
import time
from tqdm.auto import tqdm

source_file = "all_models_fewshot.xlsx"
output_file = "all_models_fewshot_llama_retried.xlsx"

retry_df = pd.read_excel(source_file)

failure_mask = (
    retry_df["LLaMA Output"]
    .fillna("")
    .astype(str)
    .str.startswith("LLaMA Error:")
)

failed_indices = retry_df.index[failure_mask].tolist()

print(f"Failed outputs to retry: {len(failed_indices)}")

max_attempts = 5

for idx in tqdm(failed_indices, desc="Retrying failed LLaMA outputs"):
    question = retry_df.at[idx, "question"]
    replacement = None

    for attempt in range(1, max_attempts + 1):
        candidate = call_llama(build_prompt(question))

        if not str(candidate).startswith("LLaMA Error:"):
            replacement = candidate
            break

        wait_seconds = 30 * attempt
        print(
            f"Row {idx}: attempt {attempt} failed. "
            f"Waiting {wait_seconds} seconds before retrying."
        )
        time.sleep(wait_seconds)

    if replacement is not None:
        retry_df.at[idx, "LLaMA Output"] = replacement
        print(f"Row {idx}: recovered successfully.")
    else:
        print(f"Row {idx}: still failed after {max_attempts} attempts.")

    # Save progress after every failed-question attempt.
    retry_df.to_excel(output_file, index=False)

remaining_failures = (
    retry_df["LLaMA Output"]
    .fillna("")
    .astype(str)
    .str.startswith("LLaMA Error:")
    .sum()
)

print(f"\nSaved {output_file}")
print(f"Remaining failed LLaMA outputs: {remaining_failures}")

In [ ]:
#all_models.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], inplace=True)

In [ ]:
all_models.rename(columns={"LLaMA Output":"LLaMA", "Allam Output": "ALLaM", "GPT-4o Output":"GPT-4o"}, inplace=True)

In [ ]:
all_models.columns

Index(['question', 'GPT-4o', 'LLaMA', 'ALLaM'], dtype='object')

In [ ]:
import json
import re
from ast import literal_eval
import pandas as pd

# List of all expected entity keys
expected_keys = ["Symptom", "Diagnosis", "Treatment", "Drug", "Dosage", "Duration", "Strength", "Unit", "Risk"]

def clean_model_output(text):
    """
    Cleans and normalizes model output (like Allam or LLaMA) into a JSON list
    with expected medical keys.
    """
    try:
        if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

        # Remove prefix text before the first "[" (e.g., "الإخراج:")
        text = text.strip()
        text = re.sub(r'^.*?\[', '[', text, flags=re.DOTALL)

        # Extract JSON-like list string
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if not match:
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

        json_str = match.group(0)

        # Try to parse using ast.literal_eval first, fallback to json.loads
        try:
            parsed = literal_eval(json_str)
        except Exception:
            parsed = json.loads(json_str)

        # Normalize each entry to include all keys
        if isinstance(parsed, list):
            cleaned = []
            for entry in parsed:
                if isinstance(entry, dict):
                    cleaned_dict = {k: str(entry.get(k, "")).strip() for k in expected_keys}
                    cleaned.append(cleaned_dict)
            return json.dumps(cleaned, ensure_ascii=False)
        else:
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

    except Exception:
        return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

In [ ]:
df = all_models.copy()

In [ ]:
import re

def clean_allam_output(text: str) -> str:
    # Normalize
    s = str(text).strip()

    # Remove Arabic/English "Output:" prefixes if present
    s = re.sub(r"^(?:الإخراج|Output)\s*:\s*", "", s, flags=re.IGNORECASE)

    if not s:
        return "no answer"

    # 1) If there is any complete JSON-like list, keep only the FIRST one
    # (and drop anything after its proper closing)
    m = re.search(r"\[\s*\{.*?\}\s*\]", s, flags=re.DOTALL)
    if m:
        return m.group(0)

    # 2) Otherwise, look for a trailing, *started* list and auto-close it.
    #    We take the LAST '[{' to avoid earlier false starts inside the text.
    start_idx = s.rfind("[{")
    if start_idx != -1:
        candidate = s[start_idx:].strip()

        # If we can find a proper closing inside the tail, cut at the first one.
        close_match = re.search(r"\}\s*\]", candidate, flags=re.DOTALL)
        if close_match:
            end = close_match.end()
            return candidate[:end]

        # Not properly closed -> auto-close
        return candidate + "}]"

    # 3) No list at all → no answer
    return "no answer"

In [ ]:
df["ALLaM Cleaned"] = df["ALLaM"].apply(clean_allam_output)

In [ ]:
from google.colab import files
df.to_excel("all_cleaned.xlsx", index=False, engine='openpyxl')
files.download("all_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df["LLaMA Cleaned"] = df["LLaMA"].apply(clean_model_output)
df["GPT-4o Cleaned"] = df["GPT-4o"].apply(clean_model_output)

In [ ]:
df.to_excel("all_cleaned.xlsx", index=False, engine='openpyxl')

In [ ]:
from google.colab import files
files.download("all_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.head()

,question,GPT-4o,LLaMA,ALLaM,ALLaM Cleaned,LLaMA Cleaned,GPT-4o Cleaned
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","الإخراج:\n[{""Symptom"": ""الخوف من عدم الإنجاب م...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً, م...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","[{""Symptom"": ""أذي نفسي، افكار بإنهاء حياتي"", ""...","الإجابة: \n[{""Symptom"": ""أذي نفسي"", ""Diagnosis...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","[{""Symptom"": ""أذي نفسي، افكار بإنهاء حياتي"", ""...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة، اصاب بلاحباط"",...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","الإخراج:\n[{""Symptom"": ""تكرار كلمة معينة لمدة ...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","[{""Symptom"": ""تكرار كلمة معينة، اصاب بلاحباط"",..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا من المجتمع، انعزال...","[{""Symptom"": ""انعزال، انطوائية، تعامل بسلبية"",...","الإخراج:\n[{""Symptom"": ""اكتئاب، فوبيا من المجت...","[{""Symptom"": ""اكتئاب، فوبيا من المجتمع، انعزال...","[{""Symptom"": ""انعزال، انطوائية، تعامل بسلبية"",...","[{""Symptom"": ""اكتئاب، فوبيا من المجتمع، انعزال..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير"", ""Diagn...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","الإخراج:\n[{""Symptom"": ""قلق شديد مزمن، خوف كبي...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير"", ""Diagn..."


##  Import Libraries & Load the data

In [ ]:
!pip install openai tqdm

In [ ]:
import pandas as pd
import openai
import json
from tqdm import tqdm
import os

In [ ]:
len(df)

499

In [ ]:
df.tail()

,question,GPT-4o,LLaMA,ALLaM,ALLaM Cleaned,LLaMA Cleaned,GPT-4o Cleaned
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","الإخراج:\n[{""Symptom"": ""قلق، توتر، وسواس قهري،...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الخوف، تنبؤات"", ""Diagnosis"": """",...","الإخراج:\n[{""Symptom"": ""الخوف، الخوف وتنبؤات م...","[{""Symptom"": ""الخوف، الخوف وتنبؤات من كلام اهل...","[{""Symptom"": ""الخوف، تنبؤات"", ""Diagnosis"": """",...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","الإخراج:\n[{""Symptom"": ""علاج من الذهانية"", ""Di...","[{""Symptom"": ""علاج من الذهانية"", ""Diagnosis"": ...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، أحس بالنوم وقت ال...","[{""Symptom"": ""فقدان التركيز، النوم وقت العمل، ...","الإخراج:\n[{""Symptom"": ""فقدان التركيز، صعوبة ا...","[{""Symptom"": ""فقدان التركيز، صعوبة التذكر، الت...","[{""Symptom"": ""فقدان التركيز، النوم وقت العمل، ...","[{""Symptom"": ""فقدان التركيز، أحس بالنوم وقت ال..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه، ماعندي شغف، مالي نف...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","الإخراج:\n[{""Symptom"": ""الاكتئاب الجزئي، عسر ا...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، فين...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","[{""Symptom"": ""ماعندي شهيه، ماعندي شغف، مالي نف..."


In [ ]:
df.columns

Index(['question', 'GPT-4o', 'LLaMA', 'ALLaM', 'ALLaM Cleaned',
       'LLaMA Cleaned', 'GPT-4o Cleaned'],
      dtype='object')

## Define the Judge Prompt

In [ ]:
def build_prompt(question_text, ner_output):
    return f"""
    You are an expert medical evaluator. Your task is to act as an impartial judge and evaluate the quality of named entity extraction performed by a single AI model. We will provide you with a medical question,
    a model-generated list of extracted entities, and a set of evaluation criteria. You should first read the question carefully, then evaluate the extracted entities based on the following two criteria: Correctness and Completeness.

    For each criterion:
    - Use only the question text as the source of truth.
    - Refer strictly to the entity definitions provided.
    - Do not infer or hallucinate missing information.
    - Output your final verdict using one of the valid options for each criterion (see below).
    - Provide a short explanation for your decision.

    ## Question:
    {question_text}

    ## Model Output (NER Prediction):
    {ner_output}

    ## Entity Schema:
    Evaluate only the following entity types and definitions:
    - Symptom: Observable psychological or behavioral symptoms (e.g., anxiety, insomnia).
    - Diagnosis: Mentioned or diagnosed mental health disorders (e.g., generalized anxiety disorder).
    - Treatment: Non-pharmacological therapy methods (e.g., cognitive behavioral therapy).
    - Drug: Names of medications or pharmaceutical substances (e.g., Xanax, Sertraline).
    - Dosage: Dosage instructions (e.g., twice a day, 50 mg).
    - Duration: Duration of medication use or symptom persistence (e.g., for one week, for 3 months).
    - Strength: Concentration of the medication (e.g., 500, 1.0).
    - Unit: Unit of measurement for strength (e.g., mg, ml, g).
    - Risk: Indicators of psychological or physical risk (e.g., suicidal thoughts, self-harm).

    ## Evaluation Criteria

    [The Start of Correctness Criterion]
    Does the model correctly extract the entity spans and assign the correct entity labels **based only on the question text**?

    - accurate: Most or all entities are correct, with minor or no issues.
    - mixed: Some correct extractions alongside boundary or label errors.
    - poor: Many incorrect, misaligned, mislabeled, or spurious entities.
    [The End of Correctness Criterion]

    [The Start of Completeness Criterion]
    How well did the model cover all important entities mentioned **in the question**?

    - full: All relevant entities present in the question are extracted.
    - partial: Some entities are extracted (about half or more).
    - low: Less than half of the important entities are captured.
    [The End of Completeness Criterion]

    ## Output Format
    Please return your evaluation in the following exact format as a markdown-style JSON block:

    {{
      "correctness": {{
        "verdict": "accurate/mixed/inaccurate",
        "reason": "Your detailed reasoning here..."
      }},
      "completeness": {{
        "verdict": "full/partial/missing",
        "reason": "Your detailed reasoning here..."
      }},
      "notes": "Optional comments if needed"
    }}

    """


## DeepSeek

In [ ]:
import os
os.environ["DEEPSEEK_API_KEY"] = "key"

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {os.getenv('DEEPSEEK_API_KEY')}",
    "Content-Type": "application/json"
}
payload = {
    "model": "deepseek-chat",
    "messages": [{"role": "user", "content": "مرحبا"}],
    "max_tokens": 10
}

try:
    r = requests.post("https://api.deepseek.com/chat/completions", headers=headers, json=payload, timeout=30)
    print(r.status_code)
    print(r.text)
except Exception as e:
    print("Error:", e)

In [ ]:
import re
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

def score_with_deepseek(prompt):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0,

            # Added for compatibility with the current DeepSeek API.
            # JSON mode forces valid JSON output and avoids parsing failures
            # caused by malformed or nested JSON responses.
            response_format={"type": "json_object"},
            extra_body={
                "thinking": {"type": "disabled"}
            }
        )

        content = response.choices[0].message.content.strip()

        if not content:
            raise ValueError("DeepSeek returned empty content")

        # JSON mode returns the full JSON object directly.
        return json.loads(content)

    except Exception as e:
        return {
            "correctness": {"verdict": "error", "reason": str(e)},
            "completeness": {"verdict": "error", "reason": str(e)},
            "notes": ""
        }

In [ ]:
import inspect

print(inspect.getsource(score_with_deepseek))

def score_with_deepseek(prompt):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0,

            # Added for compatibility with the current DeepSeek API.
            # JSON mode forces valid JSON output and avoids parsing failures
            # caused by malformed or nested JSON responses.
            response_format={"type": "json_object"},
            extra_body={
                "thinking": {"type": "disabled"}
            }
        )

        content = response.choices[0].message.content.strip()

        if not content:
            raise ValueError("DeepSeek returned empty content")

        # JSON mode returns the full JSON object directly.
        return json.loads(content)

    except Exception as e:
        return {
            "correctness": {"verdict": "error", "reason": str(e)},
            "completeness": {"verdict":

In [ ]:
# Iterate and collect results
model_cols = ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]

In [ ]:
from collections import Counter

# Reset before each new run
summary = {col: {"correctness": Counter(), "completeness": Counter()} for col in model_cols}
results = {col: [] for col in model_cols}

In [ ]:
from tqdm import tqdm

for col in model_cols:
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Scoring {col}"):
        prompt = build_prompt(row['question'], row[col])
        score = score_with_deepseek(prompt)

        results[col].append(score)

        # ✅ Update verdict counts
        if isinstance(score, dict):
            c = score.get("correctness", {}).get("verdict", "error")
            cm = score.get("completeness", {}).get("verdict", "error")
            summary[col]["correctness"][c] += 1
            summary[col]["completeness"][cm] += 1

In [ ]:
results_dict = {}

## LLaMA

In [ ]:
from tqdm import tqdm
import json

model_col = "LLaMA Cleaned"
judgment_field = "LLaMA Judgment"


for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {
            model_col: row[model_col]
        }

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save after full column is done
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)


Evaluating LLaMA Cleaned: 100%|██████████| 499/499 [16:04<00:00,  1.93s/it]


In [ ]:
llama = pd.DataFrame(results_dict.values())

In [ ]:
for question, data in results_dict.items():
    judgment = data.get("LLaMA Judgment", {})

    if judgment.get("correctness", {}).get("verdict") == "error":
        print("Question:")
        print(question)
        print("\nError reason:")
        print(judgment["correctness"]["reason"])
        break

Question:
السلام عليكم دكتور لو سمحت احس في شي يمشي في راسي ورحت عند كذا دكتور وفتش شعري قالي مافي شي وانا نفسيتي تعبانه جدا من هالموضوع ممكن تساعدني وتشرح حالتي

Error reason:
Unterminated string starting at: line 8 column 15 (char 495)


In [ ]:
llama.tail()

,LLaMA Cleaned,LLaMA Judgment
494,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,"[{""Symptom"": ""الخوف، تنبؤات"", ""Diagnosis"": """",...","{'correctness': {'verdict': 'mixed', 'reason':..."
496,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':..."
497,"[{""Symptom"": ""فقدان التركيز، النوم وقت العمل، ...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,"[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
llama.insert(0, 'Question', df['question'])

In [ ]:
llama.head()

,Question,LLaMA Cleaned,LLaMA Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي، افكار بإنهاء حياتي"", ""...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة من ا...","{'correctness': {'verdict': 'mixed', 'reason':..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""انعزال، انطوائية، تعامل بسلبية"",...","{'correctness': {'verdict': 'accurate', 'reaso..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
llama.to_excel("llama_judgments.xlsx", index=False)

from google.colab import files
files.download("llama_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
llama.to_csv("llama_judgments.csv", index=False)
files.download("llama_judgments.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

flattened = []

# Loop through each question entry in results_dict
for question, data in results_dict.items():
    for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
        # Find the corresponding judgment field for that model
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()

        if judgment_field in data:
            score = data[judgment_field]
            if isinstance(score, dict):
                flattened.append({
                    "Question": question,
                    "Model": model_col,
                    "Correctness": score["correctness"]["verdict"],
                    "Correctness Reason": score["correctness"]["reason"],
                    "Completeness": score["completeness"]["verdict"],
                    "Completeness Reason": score["completeness"]["reason"],
                    "Notes": score.get("notes", "")
                })

# Create a DataFrame and export
df_results = pd.DataFrame(flattened)
df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

Exported 499 rows to NER_Evaluation_by_DeepSeek.xlsx


In [ ]:
df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The only relevant entity in the question is th...,The model output includes empty fields for oth...
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured key entities such as self-h...,The model's output groups multiple entities in...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,mixed,The Symptom extraction correctly captures the ...,partial,"The model extracted the main symptom, the diag...",The treatment label is debatable; 'استغفري' is...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified 'اكتئاب' and 'ف...,full,All entities mentioned in the question were ex...,The model grouped multiple entities into singl...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,mixed,The model correctly identified 'الوسواس القهري...,partial,"The model captured key symptoms (anxiety, fear...",The model's extraction of 'أدوية' as Treatment...


In [ ]:
len(df_results)

499

In [ ]:
files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Allam

In [ ]:
model_col = "ALLaM Cleaned"
judgment_field = "ALLaM Judgment"

# Keep the same results_dict if already defined
for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {model_col: row[model_col]}
    else:
        results_dict[key][model_col] = row[model_col]

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Evaluating ALLaM Cleaned: 100%|██████████| 499/499 [17:38<00:00,  2.12s/it]


In [ ]:
import json
from tqdm import tqdm

model_col = "ALLaM Cleaned"
judgment_field = "ALLaM Judgment"

# Load existing results
with open("ner_judgment_results.json", "r", encoding="utf-8") as f:
    results_dict = json.load(f)

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Re-evaluating {model_col}"):
    key = row["question"]

    # Initialize model output in results if missing
    if key not in results_dict:
        results_dict[key] = {}
    results_dict[key][model_col] = row[model_col]

    # Always re-run ALLaM's judgment
    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save updated JSON
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Re-evaluating ALLaM Cleaned: 100%|██████████| 499/499 [17:33<00:00,  2.11s/it]


In [ ]:
from google.colab import files
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
allam = pd.DataFrame(results_dict.values())

In [ ]:
allam.insert(0, 'Question', df['question'])

In [ ]:
allam.tail()

,Question,GPT-4o Cleaned,GPT-4o Judgment,ALLaM Cleaned,ALLaM Judgment
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الخوف، الخوف وتنبؤات من كلام اهل...","{'correctness': {'verdict': 'poor', 'reason': ..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""علاج من الذهانية"", ""Diagnosis"": ...","{'correctness': {'verdict': 'poor', 'reason': ..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، أحس بالنوم وقت ال...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""فقدان التركيز، صعوبة التذكر، الت...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه، ماعندي شغف، مالي نف...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، فين...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
allam.to_excel("final_judgments.xlsx", index=False)

from google.colab import files
files.download("final_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
allam.to_csv("allam_judgments.csv", index=False)
files.download("allam_judgments.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## GPT-4o

In [ ]:
model_col = "GPT-4o Cleaned"
judgment_field = "GPT-4o Judgment"

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {model_col: row[model_col]}
    else:
        results_dict[key][model_col] = row[model_col]

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Evaluating GPT-4o Cleaned: 100%|██████████| 499/499 [16:21<00:00,  1.97s/it]


In [ ]:
gpt = pd.DataFrame(results_dict.values())

In [ ]:
gpt.insert(0, 'Question', df['question'])

In [ ]:
gpt.head()

,Question,GPT-4o Cleaned,GPT-4o Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة، اصاب بلاحباط"",...","{'correctness': {'verdict': 'mixed', 'reason':..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""اكتئاب، فوبيا من المجتمع، انعزال...","{'correctness': {'verdict': 'accurate', 'reaso..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير"", ""Diagn...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [ ]:
test_prompt = build_prompt(
    df.iloc[0]["question"],
    df.iloc[0]["GPT-4o Cleaned"]
)

test_result = score_with_deepseek(test_prompt)

test_result

{'correctness': {'verdict': 'accurate',
  'reason': "The model correctly identified 'الخوف من عدم الإنجاب' (fear of not having children) as a Symptom, which aligns with the definition of observable psychological symptoms. No other entities from the schema are present in the question, so no mislabeling or spurious extractions occurred."},
 'completeness': {'verdict': 'full',
  'reason': 'The only relevant entity in the question is the symptom of fear regarding future childlessness. The model extracted this completely, and no other entity types (Diagnosis, Treatment, Drug, etc.) are mentioned in the question.'},
 'notes': 'The extraction is accurate and complete given the limited entities present in the question.'}

In [ ]:
gpt.to_excel("gpt_judgments.xlsx", index=False)

from google.colab import files
files.download("gpt_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
gpt.to_csv("gpt_judgments.csv", index=False)
files.download("gpt_judgments.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd


# Loop through each question entry in results_dict
for question, data in results_dict.items():
    for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
        # Find the corresponding judgment field for that model
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()

        if judgment_field in data:
            score = data[judgment_field]
            if isinstance(score, dict):
                flattened.append({
                    "Question": question,
                    "Model": model_col,
                    "Correctness": score["correctness"]["verdict"],
                    "Correctness Reason": score["correctness"]["reason"],
                    "Completeness": score["completeness"]["verdict"],
                    "Completeness Reason": score["completeness"]["reason"],
                    "Notes": score.get("notes", "")
                })

# Create a DataFrame and export
df_results = pd.DataFrame(flattened)
df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

Exported 998 rows to NER_Evaluation_by_DeepSeek.xlsx


In [ ]:
import json
from ast import literal_eval
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

# Files containing the completed judgments.
gpt_allam_file = "gpt-allam_judgments.csv"
llama_file = "ner_judgment_results.json"

# Final combined Excel file.
output_file = "few_shot_NER_Evaluation_by_DeepSeek_retried.xlsx"


# Load the wide CSV containing GPT-4o and ALLaM results.
gpt_allam_df = pd.read_csv(gpt_allam_file, encoding="utf-8-sig")

# Load LLaMA Judge results from JSON.
with open(llama_file, "r", encoding="utf-8") as f:
    llama_results = json.load(f)

# Convert LLaMA JSON results into a table.
llama_rows = []

for question, data in llama_results.items():
    llama_rows.append({
        "Question": question,
        "LLaMA Cleaned": data.get("LLaMA Cleaned", ""),
        "LLaMA Judgment": json.dumps(
            data.get("LLaMA Judgment", {}),
            ensure_ascii=False
        )
    })

llama_df = pd.DataFrame(llama_rows)

# Merge the corrected LLaMA columns with GPT-4o and ALLaM columns.
final_df = gpt_allam_df.merge(
    llama_df,
    on="Question",
    how="left",
    validate="one_to_one"
)

# Export the combined file.
final_df.to_excel(output_file, index=False)

# Apply consistent formatting.
workbook = load_workbook(output_file)
worksheet = workbook.active
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions

for row in worksheet.iter_rows():
    for cell in row:
        cell.font = Font(name="Arial", size=10)

for cell in worksheet[1]:
    cell.font = Font(name="Arial", size=10, bold=True)

workbook.save(output_file)

print(f"Saved: {output_file}")
print(f"Rows: {len(final_df)}")
print("\nColumns:")
print(final_df.columns.tolist())

print("\nMissing LLaMA values:")
print(
    final_df[
        ["LLaMA Cleaned", "LLaMA Judgment"]
    ].isna().sum()
)

Saved: few_shot_NER_Evaluation_by_DeepSeek_retried.xlsx
Rows: 499

Columns:
['Question', 'GPT-4o Cleaned', 'GPT-4o Judgment', 'ALLaM Cleaned', 'ALLaM Judgment', 'LLaMA Cleaned', 'LLaMA Judgment']

Missing LLaMA values:
LLaMA Cleaned     0
LLaMA Judgment    0
dtype: int64


In [ ]:
import json
from ast import literal_eval

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

input_file = "few_shot_NER_Evaluation_by_DeepSeek_retried.xlsx"
output_file = "NER_Evaluation_by_DeepSeek_flattened_retried.xlsx"


def parse_judgment(value):
    """Convert an Excel judgment cell into a Python dictionary."""
    if isinstance(value, dict):
        return value

    if pd.isna(value) or str(value).strip() == "":
        return None

    text = str(value).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            return literal_eval(text)
        except (ValueError, SyntaxError):
            return None


# Load the combined wide-format file.
judge_df = pd.read_excel(input_file)

# Start with an empty list every time to avoid duplicate rows.
flattened = []

for _, row in judge_df.iterrows():
    question = row["Question"]

    for model_col in [
        "LLaMA Cleaned",
        "ALLaM Cleaned",
        "GPT-4o Cleaned"
    ]:
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()
        score = parse_judgment(row[judgment_field])

        if isinstance(score, dict):
            correctness = score.get("correctness", {})
            completeness = score.get("completeness", {})

            flattened.append({
                "Question": question,
                "Model": model_col,
                "Correctness": correctness.get("verdict", ""),
                "Correctness Reason": correctness.get("reason", ""),
                "Completeness": completeness.get("verdict", ""),
                "Completeness Reason": completeness.get("reason", ""),
                "Notes": score.get("notes", "")
            })

# Create and export the long-format evaluation table.
df_results = pd.DataFrame(flattened)
df_results.to_excel(output_file, index=False)

# Apply simple formatting.
workbook = load_workbook(output_file)
worksheet = workbook.active
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions

for row in worksheet.iter_rows():
    for cell in row:
        cell.font = Font(name="Arial", size=10)

for cell in worksheet[1]:
    cell.font = Font(name="Arial", size=10, bold=True)

workbook.save(output_file)

print(f"Exported {len(df_results)} rows to {output_file}")
print(df_results["Model"].value_counts())

Exported 1497 rows to NER_Evaluation_by_DeepSeek_flattened_retried.xlsx
Model
LLaMA Cleaned     499
ALLaM Cleaned     499
GPT-4o Cleaned    499
Name: count, dtype: int64


In [ ]:
import json
import pandas as pd

input_file = "NER_Evaluation_by_DeepSeek_flattened_retried.xlsx"
json_file = "NER_Evaluation_by_DeepSeek_grouped_by_question.json"

df_results = pd.read_excel(input_file)

grouped_results = []

for question, group in df_results.groupby("Question", sort=False):
    question_record = {
        "Question": question
    }

    for _, row in group.iterrows():
        model_name = str(row["Model"]).replace(" Cleaned", "")

        question_record[model_name] = {
            "Correctness": row["Correctness"],
            "Correctness Reason": row["Correctness Reason"],
            "Completeness": row["Completeness"],
            "Completeness Reason": row["Completeness Reason"],
            "Notes": row["Notes"]
        }

    grouped_results.append(question_record)

with open(json_file, "w", encoding="utf-8") as f:
    json.dump(
        grouped_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved: {json_file}")
print(f"Questions: {len(grouped_results)}")
print("Models in the first record:", list(grouped_results[0].keys()))

Saved: NER_Evaluation_by_DeepSeek_grouped_by_question.json
Questions: 499
Models in the first record: ['Question', 'LLaMA', 'ALLaM', 'GPT-4o']


In [ ]:
df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The only relevant entity in the question is th...,The model output includes empty fields for oth...
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured key entities such as self-h...,The model's output groups multiple entities in...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,mixed,The Symptom extraction correctly captures the ...,partial,"The model extracted the main symptom, the diag...",The treatment label is debatable; 'استغفري' is...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified 'اكتئاب' and 'ف...,full,All entities mentioned in the question were ex...,The model grouped multiple entities into singl...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,mixed,The model correctly identified 'الوسواس القهري...,partial,"The model captured key symptoms (anxiety, fear...",The model's extraction of 'أدوية' as Treatment...


In [ ]:
df_results['Model'].unique()

array(['LLaMA Cleaned'], dtype=object)

In [ ]:
files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Check missing Q form the Judge

In [ ]:
import json
from ast import literal_eval

import pandas as pd

wide_file = "few_shot_NER_Evaluation_by_DeepSeek_retried.xlsx"
output_file = "few_shot_ner_judgment_results_repaired.json"

wide_df = pd.read_excel(wide_file)


def parse_judgment(value):
    """Convert an Excel judgment cell into a dictionary."""
    if isinstance(value, dict):
        return value

    if pd.isna(value) or str(value).strip() == "":
        return {}

    text = str(value).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            return literal_eval(text)
        except (ValueError, SyntaxError):
            return {}


# Rebuild a clean published-style results dictionary.
results_dict = {}

for _, row in wide_df.iterrows():
    question = row["Question"]

    results_dict[question] = {
        "GPT-4o Cleaned": row["GPT-4o Cleaned"],
        "GPT-4o Judgment": parse_judgment(row["GPT-4o Judgment"]),
        "ALLaM Cleaned": row["ALLaM Cleaned"],
        "ALLaM Judgment": parse_judgment(row["ALLaM Judgment"]),
        "LLaMA Cleaned": row["LLaMA Cleaned"],
        "LLaMA Judgment": parse_judgment(row["LLaMA Judgment"])
    }


targets = [
    {
        "model_col": "ALLaM Cleaned",
        "judgment_field": "ALLaM Judgment",
        "question": (
            "عدم تركيز مع الآخرين وتوتر واختناق وعدم القدرة على النوم "
            "وساوس وزن رهيب في الدماغ والتوتر الدائم"
        )
    },
    {
        "model_col": "LLaMA Cleaned",
        "judgment_field": "LLaMA Judgment",
        "question": (
            "السلام عليكم دكتور لو سمحت احس في شي يمشي في راسي ورحت عند كذا "
            "دكتور وفتش شعري قالي مافي شي وانا نفسيتي تعبانه جدا من هالموضوع "
            "ممكن تساعدني وتشرح حالتي"
        )
    }
]

for target in targets:
    question = target["question"]
    model_col = target["model_col"]
    judgment_field = target["judgment_field"]

    matched = wide_df[
        wide_df["Question"].astype(str).str.strip() == question.strip()
    ]

    if len(matched) != 1:
        raise RuntimeError(
            f"Expected one match for {judgment_field}, found {len(matched)}."
        )

    row = matched.iloc[0]

    prompt = build_prompt(row["Question"], row[model_col])
    score = score_with_deepseek(prompt)

    if score.get("correctness", {}).get("verdict") == "error":
        print(f"Retry failed: {judgment_field}")
        print(score)
        continue

    results_dict[question][judgment_field] = score

    print(f"Recovered: {judgment_field}")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results_dict, f, ensure_ascii=False, indent=2)

# Final validation.
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

for field in ["GPT-4o Judgment", "ALLaM Judgment", "LLaMA Judgment"]:
    valid_count = sum(
        isinstance(record.get(field), dict)
        and record[field].get("correctness", {}).get("verdict") != "error"
        for record in results_dict.values()
    )
    print(f"{field}: {valid_count} / {len(results_dict)}")

print(f"Saved: {output_file}")

Recovered: ALLaM Judgment
Recovered: LLaMA Judgment
GPT-4o Judgment: 499 / 499
ALLaM Judgment: 499 / 499
LLaMA Judgment: 499 / 499
Saved: few_shot_ner_judgment_results_repaired.json


# Error Analysis

In [ ]:
import json
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# Final judgment files
FEW_FILE = "few_shot_ner_judgment_results_repaired.json"
ZERO_FILE = "zero_shot_ner_judgment_results_retried.json"
OUTPUT_FILE = "LLM_as_Judge_error_analysis_review.xlsx"

MODELS = ["GPT-4o", "LLaMA", "ALLaM"]

# Map alternative verdict words to the paper's three-level scale
VERDICT_MAP = {
    "correctness": {"inaccurate": "poor"},
    "completeness": {"missing": "low"}
}

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError(f"{path} must contain a dictionary keyed by question.")

    return data

def normalise_verdict(judgment, criterion):
    block = judgment.get(criterion, {}) if isinstance(judgment, dict) else {}
    verdict = str(block.get("verdict", "")).strip().lower()
    return VERDICT_MAP[criterion].get(verdict, verdict or "missing_judgment")

def get_reason(judgment, criterion):
    block = judgment.get(criterion, {}) if isinstance(judgment, dict) else {}
    return str(block.get("reason", "")).strip()

def serialise_output(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return "" if value is None else str(value)

def build_judge_dataframe(results_dict, prompting):
    rows = []

    for question, data in results_dict.items():
        for model in MODELS:
            judgment = data.get(f"{model} Judgment", {})
            correctness = normalise_verdict(judgment, "correctness")
            completeness = normalise_verdict(judgment, "completeness")

            # High priority: poor correctness and/or low completeness
            if correctness == "poor" or completeness == "low":
                priority = "high"
            # Medium priority: mixed correctness and/or partial completeness
            elif correctness == "mixed" or completeness == "partial":
                priority = "medium"
            else:
                priority = "none"

            rows.append({
                "Prompting": prompting,
                "Model": model,
                "Question": question,
                "Model Output": serialise_output(data.get(f"{model} Cleaned", "")),
                "Correctness Verdict": correctness,
                "Correctness Reason": get_reason(judgment, "correctness"),
                "Completeness Verdict": completeness,
                "Completeness Reason": get_reason(judgment, "completeness"),
                "Judge Notes": str(judgment.get("notes", "")).strip()
                    if isinstance(judgment, dict) else "",
                "Review Priority": priority,
                "Manual Error Category": "",
                "Manual Evidence": "",
                "Agree with Judge?": ""
            })

    return pd.DataFrame(rows)

few_df = build_judge_dataframe(load_json(FEW_FILE), "Few-shot")
zero_df = build_judge_dataframe(load_json(ZERO_FILE), "Zero-shot")
all_df = pd.concat([few_df, zero_df], ignore_index=True)

# Keep only outputs that DeepSeek flagged for review
review_df = all_df[all_df["Review Priority"] != "none"].copy()

# Quantitative summary of judge-flagged outputs
summary_df = (
    all_df.assign(
        Non_Accurate=all_df["Correctness Verdict"].ne("accurate"),
        Non_Full=all_df["Completeness Verdict"].ne("full"),
        High_Priority=all_df["Review Priority"].eq("high")
    )
    .groupby(["Prompting", "Model"], as_index=False)
    .agg(
        Total_Items=("Question", "size"),
        Non_Accurate_Items=("Non_Accurate", "sum"),
        Non_Full_Items=("Non_Full", "sum"),
        High_Priority_Items=("High_Priority", "sum"),
        Judge_Flagged_Items=("Review Priority", lambda s: (s != "none").sum())
    )
)

# Draw up to five high-priority and five medium-priority cases per condition/model
sample_parts = []
for _, group in review_df.groupby(["Prompting", "Model", "Review Priority"]):
    sample_parts.append(
        group.sample(n=min(5, len(group)), random_state=42)
    )

sample_df = (
    pd.concat(sample_parts, ignore_index=True)
    .sort_values(["Prompting", "Model", "Review Priority"])
)

coding_guide_df = pd.DataFrame({
    "Manual Error Category": [
        "omission",
        "spurious_entity",
        "mislabeling",
        "boundary_or_granularity",
        "mixed_error",
        "schema_ambiguity",
        "judge_overreach",
        "other"
    ],
    "Definition": [
        "A relevant entity or mention was not extracted.",
        "The model extracted an entity not supported by the question.",
        "The extracted span was assigned to the wrong entity type.",
        "The entity is partly correct but its span is too broad, too narrow, or overly fragmented.",
        "More than one extraction error type occurs in the same output.",
        "The question or schema definition makes the decision genuinely unclear.",
        "The judge appears to infer an entity or apply the schema inconsistently.",
        "A pattern not covered by the categories above."
    ]
})

# Export a structured workbook for manual qualitative coding
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    review_df.to_excel(writer, sheet_name="All Judge-Flagged Cases", index=False)
    sample_df.to_excel(writer, sheet_name="Stratified Review Sample", index=False)
    coding_guide_df.to_excel(writer, sheet_name="Coding Guide", index=False)

    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for cell in worksheet[1]:
            cell.font = Font(name="Arial", bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="1F4E78")
            cell.alignment = Alignment(horizontal="center", vertical="center")

        for column_cells in worksheet.columns:
            max_length = min(
                max(len(str(cell.value or "")) for cell in column_cells) + 2,
                50
            )
            worksheet.column_dimensions[
                get_column_letter(column_cells[0].column)
            ].width = max_length

        for row in worksheet.iter_rows(min_row=2):
            for cell in row:
                cell.font = Font(name="Arial", size=10)
                cell.alignment = Alignment(vertical="top", wrap_text=True)

print(f"Saved: {OUTPUT_FILE}")
display(summary_df)
print(f"Judge-flagged cases for review: {len(review_df)}")
print(f"Stratified manual-review sample: {len(sample_df)}")

Saved: LLM_as_Judge_error_analysis_review.xlsx


,Prompting,Model,Total_Items,Non_Accurate_Items,Non_Full_Items,High_Priority_Items,Judge_Flagged_Items
0,Few-shot,ALLaM,499,483,411,143,483
1,Few-shot,GPT-4o,499,381,302,16,383
2,Few-shot,LLaMA,499,384,303,16,386
3,Zero-shot,ALLaM,499,457,445,198,473
4,Zero-shot,GPT-4o,499,383,381,106,405
5,Zero-shot,LLaMA,499,370,296,54,376


Judge-flagged cases for review: 2506
Stratified manual-review sample: 60
